# Bài 04 · Softmax & hàm mất mát: biến 10 điểm số thành xác suất và đo mạng sai bao nhiêu

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/04-ham-mat-mat.ipynb)

Notebook đi kèm [Bài 04 của Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/ham-mat-mat/) (demo tương tác nằm ở trang bài học).
Mạng MNIST của khóa học, 784 → 128 → 64 → 10, cho mỗi ảnh 10 điểm số (**logit**), mỗi chữ số một điểm.
Ta tự viết bằng numpy softmax để biến chúng thành xác suất, cross-entropy và MSE để đo mạng sai bao nhiêu,
rồi huấn luyện lại đúng mạng mà demo dùng trên 60.000 ảnh MNIST thật.

**Bạn sẽ làm:**
1. Biến 10 logit thành 10 xác suất bằng softmax, và viết nó sao cho không bao giờ tràn số.
2. Tính cross-entropy thẳng từ logit, và hiểu vì sao mạng chưa học có mất mát gần $\ln 10$.
3. Huấn luyện lại mạng của khóa học bằng cross-entropy, ra đúng 97,27% như demo.
4. So MSE với cross-entropy trên những ảnh “tự tin mà sai”: bằng độ dốc, rồi bằng huấn luyện thật.
5. Xem xác suất của softmax đáng tin đến đâu, và nhìn mất mát như một địa hình cho Bài 05.

**Cần biết trước:** [Bài 01 · Perceptron](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/perceptron/) ($z = xW + b$),
[Bài 02 · Hàm kích hoạt](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/ham-kich-hoat/) (ReLU) và
[Bài 03 · Mạng nhiều lớp & bài toán XOR](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/mlp-va-xor/) (lớp ẩn).
**Thời gian chạy:** dưới 1 phút trên CPU, kể cả tải MNIST (~11 MB) lần đầu.

Ký hiệu theo chương *Neural Computation* của [MLSysBook](https://mlsysbook.ai/vol1/nn_computation/nn_computation.html),
với vectơ hàng như Bài 01: $z = h W + b$. Mọi con số trong notebook đều do chính các ô bên dưới tính ra,
và trùng với số trong demo.

> Các ô **🤔 Dự đoán trước** đặt một câu hỏi trước khi chạy. Hãy tự trả lời rồi mới chạy ô kế tiếp.

## 1 · Chuẩn bị

Chỉ cần numpy và matplotlib (Colab đã cài sẵn). Hàm `vn()` in số theo kiểu Việt Nam: dấu phẩy thập phân,
dấu chấm hàng nghìn. Python vẫn *gõ* số thập phân bằng dấu chấm (`0.1`), chỉ phần in ra là đổi.

In [ ]:
import gzip
import hashlib
import itertools
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FuncFormatter

# Bảng màu của khóa học (an toàn cho người mù màu); luôn đi kèm một mã hóa thứ hai (nét liền/đứt, gạch chéo, nhãn)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def vn(x, digits=2):
    """Số kiểu Việt Nam: 13.007 · 0,1 · −4,38."""
    s = f"{x:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    return s.replace("-", "−")


def pct(p, digits=2):
    return vn(100 * p, digits) + "%"


def sci(x):
    """Số rất nhỏ, dạng khoa học: 1,2e−10."""
    return f"{x:.1e}".replace(".", ",").replace("-", "−")


def vn_ticks(axis, digits=1):
    """Nhãn trục của matplotlib cũng dùng dấu phẩy thập phân."""
    axis.set_major_formatter(FuncFormatter(lambda v, _: vn(v, digits)))


print(vn(60000, 0), vn(np.log(10), 4), pct(0.9727), sci(1.23e-10))

## 2 · Dữ liệu MNIST

MNIST (LeCun và cộng sự, 1998) gồm 70.000 ảnh chữ số viết tay xám 28×28: 60.000 ảnh huấn luyện, 10.000 ảnh kiểm tra.
Ô dưới tải 4 tệp gốc từ một bản sao công khai và kiểm tra mã MD5, nên ai chạy cũng có đúng cùng một dữ liệu.
Như Bài 01, mỗi ảnh được trải thành một vectơ hàng $x$ gồm 784 số trong $[0, 1]$.

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # mã MD5 chuẩn của MNIST
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"


def load(kind):
    """Định dạng IDX: 16 byte đầu của tệp ảnh (8 byte của tệp nhãn) là phần mô tả, còn lại là các byte 0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 28, 28)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images, labels


fetch()
train_img, train_lab = load("train")
test_img, test_lab = load("t10k")
Xtr, ytr = train_img.reshape(-1, 784) / 255.0, train_lab.astype(int)  # mỗi hàng là một ảnh
Xte, yte = test_img.reshape(-1, 784) / 255.0, test_lab.astype(int)
print(
    f"{vn(len(ytr), 0)} ảnh huấn luyện + {vn(len(yte), 0)} ảnh kiểm tra = {vn(len(ytr) + len(yte), 0)};  X:", Xtr.shape
)
print("nhãn:", np.unique(ytr))

Demo trên trang bài học dùng 18 ảnh kiểm tra thật, chia ba nhóm theo cách mạng đã học trả lời chúng
(chọn bởi `reel/mnist_mlp/train.py` của khóa học). Notebook dùng lại đúng 18 ảnh đó; tên nhóm sẽ có nghĩa ở mục 7,
sau khi ta tự huấn luyện mạng. Hình dưới vẽ chúng, mỗi ảnh ghi số thứ tự trong tập kiểm tra và nhãn đúng.

In [ ]:
DEMO = {  # chỉ số gốc trong tập kiểm tra MNIST, giống hệt demo
    "đúng & chắc": [0, 1, 2, 3, 4, 7],
    "đúng nhưng phân vân": [62, 340, 358, 447, 691, 890],
    "sai & tự tin": [2654, 9009, 2939, 4880, 5955, 1112],
}
fig, axes = plt.subplots(3, 6, figsize=(8.5, 4.6))
for row, (group, idx) in zip(axes, DEMO.items()):
    for ax, i in zip(row, idx):
        ax.imshow(test_img[i], cmap="gray_r")
        ax.set_title(f"#{i} · số {yte[i]}", fontsize=9)
        ax.axis("off")
    row[0].text(-0.3, 0.5, group, transform=row[0].transAxes, ha="right", va="center", fontsize=10)
plt.subplots_adjust(left=0.24, hspace=0.35)
plt.show()

## 3 · Mười điểm số: logit của mạng 784 → 128 → 64 → 10

Mạng của khóa học (MLSysBook dùng đúng mạng này cho MNIST) có hai lớp ẩn ReLU và một lớp ra 10 nơ-ron:

$$h_1 = \mathrm{ReLU}(x W_1 + b_1), \qquad h_2 = \mathrm{ReLU}(h_1 W_2 + b_2), \qquad z = h_2 W_3 + b_3 .$$

Mười số $z_0, \dots, z_9$ ở lớp ra gọi là **logit**: điểm số thô cho từng chữ số, chưa qua hàm nào.
Trọng số khởi tạo ngẫu nhiên theo cách của He (Bài 09 sẽ giải thích): $W \sim \mathcal{N}(0, 2/n_\text{vào})$, $b = 0$,
với seed 0, giống hệt chương trình đã tạo ra demo. Ô dưới in 10 logit của mạng *chưa học* cho ảnh kiểm tra #0.

In [ ]:
SIZES = [784, 128, 64, 10]


def init(seed=0, last_scale=1.0):
    """Khởi tạo He như reel/mnist_mlp/train.py. Trả về cả rng, vì lúc huấn luyện rng còn dùng để xáo ảnh."""
    rng = np.random.default_rng(seed)
    params = [
        [rng.normal(0, np.sqrt(2 / n_in), (n_in, n_out)), np.zeros(n_out)] for n_in, n_out in itertools.pairwise(SIZES)
    ]
    params[2][0] *= last_scale  # phóng to/thu nhỏ W₃ của lớp ra: chỉ dùng cho các thí nghiệm ở mục 5 và 8
    return params, rng


def forward(params, X):
    """Mỗi hàng của X là một ảnh; trả về h₁, h₂ và z (10 logit cho mỗi ảnh)."""
    (W1, b1), (W2, b2), (W3, b3) = params
    h1 = np.maximum(0, X @ W1 + b1)  # ReLU (Bài 02)
    h2 = np.maximum(0, h1 @ W2 + b2)
    return h1, h2, h2 @ W3 + b3


params0, _ = init(seed=0)
n_params = sum(W.size + b.size for W, b in params0)
z0 = forward(params0, Xte[:1])[2][0]  # ảnh kiểm tra #0
print(f"mạng có {vn(n_params, 0)} tham số; ảnh #0 là số {yte[0]}")
print("10 logit:", "  ".join(f"{k}: {vn(v, 3)}" for k, v in enumerate(z0)))
print(f"logit lớn nhất ở chữ số {z0.argmax()}; tổng 10 logit = {vn(z0.sum(), 2)}")
assert n_params == 109386  # 109.184 trọng số (đúng bằng số MAC ở Bài 01) + 202 độ lệch

Logit có thể âm, có thể dương, và tổng của chúng chẳng có ý nghĩa gì. Mạng chưa học nên chữ số được điểm cao nhất
chỉ là ngẫu nhiên. Muốn đọc 10 điểm số này như “mạng nghĩ ảnh là chữ số $k$ với xác suất bao nhiêu”, ta cần
10 số **dương** có **tổng bằng 1**.

### 🤔 Dự đoán trước
Cách đơn giản nhất: chia mỗi logit cho tổng, $z_k / \sum_j z_j$. Tổng của 10 số mới chắc chắn bằng 1.
Chúng có phải 10 xác suất không? Chữ số có logit lớn nhất (số 9) sẽ nhận “xác suất” bao nhiêu?

In [ ]:
naive = z0 / z0.sum()
print("z / Σz =", "  ".join(f"{k}: {vn(v, 3)}" for k, v in enumerate(naive)))
print(
    f"tổng = {vn(naive.sum(), 3)};  nhỏ nhất {vn(naive.min(), 3)} (chữ số {naive.argmin()});  lớn nhất {vn(naive.max(), 3)}"
)

Tổng đúng là 1, nhưng có số âm và có số lớn hơn 1: không phải xác suất. Tệ hơn, tổng 10 logit ở đây âm, nên phép chia
**đảo ngược thứ tự**: chữ số 9, có logit lớn nhất, lại nhận “xác suất” nhỏ nhất (−1,516). Ta cần một cách biến
mọi logit thành số dương trước khi chia, và giữ nguyên thứ tự: đó là hàm mũ.

## 4 · Softmax: lũy thừa rồi chia cho tổng

$$\hat y_k = \mathrm{softmax}(z)_k = \frac{e^{z_k}}{\sum_{j=0}^{9} e^{z_j}}$$

$e^{z}$ luôn dương và tăng theo $z$, nên logit lớn hơn luôn nhận xác suất lớn hơn, và không lớp nào bị ép về đúng 0.
Viết bằng vòng lặp cho thấy rõ ba bước:

In [ ]:
def softmax_loop(z):
    e = [float(np.exp(v)) for v in z]  # 1. lũy thừa: mọi số đều thành dương, thứ tự giữ nguyên
    total = sum(e)  # 2. cộng lại
    return [v / total for v in e]  # 3. chia cho tổng: tổng mới bằng 1


p0 = softmax_loop(z0)
print("softmax(z) =", "  ".join(f"{k}: {vn(v, 3)}" for k, v in enumerate(p0)))
print(f"tổng = {vn(sum(p0), 3)};  lớn nhất {vn(max(p0), 3)} (chữ số {int(np.argmax(p0))});  nhỏ nhất {vn(min(p0), 3)}")

Mạng chưa học nên 10 xác suất chỉ từ 0,057 đến 0,205, quanh 0,1: nó gần như đoán đều. Chữ số 9 vẫn đứng đầu, như thứ tự của logit.

Softmax chỉ phụ thuộc vào **hiệu** giữa các logit: cộng cùng một số $c$ vào cả 10 logit thì $e^{c}$ xuất hiện ở cả tử
lẫn mẫu và triệt tiêu. Nhân các logit với một số thì khác: hiệu giữa chúng giãn ra, xác suất lớn nhất tiến về 1.
(Vì thế Goodfellow và cộng sự cho rằng tên đúng hơn là “softargmax”: một phiên bản *mềm* của việc chọn chữ số
có điểm cao nhất.) Phiên bản numpy dưới xử lý nhiều ảnh cùng lúc, mỗi hàng một ảnh:

In [ ]:
def softmax(Z):
    """Softmax theo từng hàng, bản ổn định: trừ logit lớn nhất trước khi lũy thừa (lý do ở ô sau)."""
    E = np.exp(Z - Z.max(axis=-1, keepdims=True))
    return E / E.sum(axis=-1, keepdims=True)


assert np.allclose(softmax(z0), p0)
for name, z in [("z", z0), ("z + 100", z0 + 100), ("z − 5", z0 - 5), ("3·z", 3 * z0), ("10·z", 10 * z0)]:
    p = softmax(z)
    print(f"{name:>8}: xác suất lớn nhất {vn(p.max(), 4)} (chữ số {p.argmax()}), nhỏ nhất {vn(p.min(), 4)}")
assert np.allclose(softmax(z0 + 100), softmax(z0)) and np.allclose(softmax(z0 - 5), softmax(z0))

### Vì sao phải trừ logit lớn nhất?
Máy tính chỉ biểu diễn được số thực đến một giới hạn. $e^{1000}$ vượt quá giới hạn đó và thành `inf` (vô cùng),
và `inf / inf` là `nan` (“không phải một số”). Vì softmax không đổi khi cộng cùng một số vào mọi logit, ta trừ
logit lớn nhất trước: số mũ lớn nhất khi đó là $e^0 = 1$, không bao giờ tràn. Ô dưới so hai cách trên ba logit
rất lớn, rồi in ngưỡng tràn số của ba kiểu số thực thường gặp.

In [ ]:
z_big = np.array([1000.0, 999.0, 990.0])
with np.errstate(over="ignore", invalid="ignore"):  # tắt cảnh báo của numpy để nhìn thẳng vào kết quả
    e = np.exp(z_big)
    print("cách ngây thơ:  e^z =", e, "  →  e^z / Σe^z =", e / e.sum())
print("trừ max trước: ", "  ".join(vn(v, 6) for v in softmax(z_big)))
for t in (np.float16, np.float32, np.float64):
    print(f"{t.__name__:>8}: e^z tràn số khi z > {vn(float(np.log(np.finfo(t).max)), 2)}")

Với float64 (mặc định của numpy) phải đến $z > 709{,}78$ mới tràn, nhưng khi huấn luyện trên GPU người ta hay tính
bằng float16, nơi chỉ cần một logit lớn hơn 11,09. Ở mục 7 ta sẽ thấy logit thật của mạng đã học vượt xa ngưỡng đó.

## 5 · Cross-entropy: mức “bất ngờ” khi thấy đáp án đúng

Để học, mạng cần một con số nói nó sai đến đâu: **hàm mất mát**, càng nhỏ càng tốt. Với nhãn one-hot
($y_k = 1$ ở chữ số đúng $c$, 0 ở các chữ số khác), cross-entropy chỉ còn một số hạng:

$$\mathcal{L}_\text{CE} = -\sum_{k} y_k \ln \hat y_k = -\ln \hat y_c .$$

$-\ln \hat y_c$ là mức **bất ngờ** của mạng khi thấy đáp án: gán 0,9 cho chữ số đúng thì gần như không bất ngờ,
gán 0,0001 thì rất bất ngờ. Logarit tự nhiên (ln) đo mức này bằng đơn vị *nat*. Lấy $\ln$ của softmax có thể
viết gọn thành $\ln \hat y_k = z_k - \ln \sum_j e^{z_j}$ (**log-softmax**), nên ta tính thẳng từ logit,
cũng trừ max trước. Như thế không bao giờ phải lấy $\ln 0$.

In [ ]:
def log_softmax(Z):
    """ln softmax(Z) tính thẳng từ logit: (z − max) − ln Σ e^(z − max)."""
    Zs = Z - Z.max(axis=-1, keepdims=True)
    return Zs - np.log(np.exp(Zs).sum(axis=-1, keepdims=True))


def cross_entropy(Z, y):
    """Trung bình trên các ảnh của −ln ŷ_c, với c là chữ số đúng của từng ảnh."""
    return float(-log_softmax(Z)[np.arange(len(y)), y].mean())


c0 = yte[0]
print(f"ảnh #0 là số {c0}: ŷ_{c0} = {vn(p0[c0], 4)}  →  −ln ŷ_{c0} = {vn(cross_entropy(z0[None], yte[:1]), 4)}")
for p in [0.99, 0.9, 0.8, 0.5, 0.1, 0.01, 0.0001]:
    print(f"xác suất cho đáp án đúng {vn(p, 4):>6}  →  mất mát {vn(-np.log(p), 3)}")

Xác suất cho đáp án đúng càng gần 0, mất mát càng lớn, và không có giới hạn trên: $-\ln \hat y_c \to \infty$ khi
$\hat y_c \to 0$. (MLSysBook lấy đúng ví dụ 0,8 → 0,223 và 0,9 → 0,105.) Trên một tập dữ liệu, mất mát là trung bình
của các ảnh.

### 🤔 Dự đoán trước
Mạng *chưa học* ở mục 3. Trên cả 10.000 ảnh kiểm tra, cross-entropy trung bình gần con số nào?
Gợi ý: một mạng đoán đều cả 10 chữ số thì gán cho đáp án đúng xác suất bao nhiêu?

In [ ]:
loss0 = cross_entropy(forward(params0, Xte)[2], yte)
print(f"mất mát lúc chưa học: {vn(loss0, 4)}   ·   −ln(1/10) = ln 10 = {vn(np.log(10), 4)}")
assert abs(loss0 - 2.4257) < 5e-4, "khác với demo: hãy kiểm tra lại dữ liệu"

Đoán đều thì $\hat y_c = 1/10$ và mất mát đúng bằng $\ln 10 \approx 2{,}3026$: đó là mốc “chưa biết gì” của mọi
bài phân loại 10 lớp. Mạng chưa học được 2,4257 (đúng con số demo ghi), hơi cao hơn, vì logit ngẫu nhiên không bằng
nhau tuyệt đối nên softmax không đều hẳn. Karpathy khuyên luôn kiểm tra con số này trước khi huấn luyện: nếu mất
mát ban đầu khác xa $\ln 10$, lớp ra đang được khởi tạo quá “tự tin”. Thử phóng to/thu nhỏ $W_3$ của lớp ra:

In [ ]:
for s in [0, 0.1, 0.5, 1, 2, 4]:
    Zs = forward(init(seed=0, last_scale=s)[0], Xte)[2]
    print(
        f"W₃ × {vn(s, 1):>3}: logit lệch chuẩn {vn(Zs.std(), 2)}, mất mát {vn(cross_entropy(Zs, yte), 4)}, "
        f"trung bình −ln ŷₖ trên cả 10 lớp {vn(-log_softmax(Zs).mean(), 4)}"
    )
    if s == 0:
        assert np.isclose(cross_entropy(Zs, yte), np.log(10))  # W₃ = 0: mọi logit bằng 0, đoán đều tuyệt đối

Với $W_3 = 0$, mọi logit bằng 0 và mất mát đúng bằng $\ln 10$. Logit càng tản mạn, mạng càng “tự tin” một cách
ngẫu nhiên và mất mát ban đầu càng cao: 3,3089 khi $W_3$ lớn gấp 4. Cột cuối là trung bình $-\ln \hat y_k$ trên
cả 10 lớp, không chỉ lớp đúng. Nó không bao giờ dưới $\ln 10$, vì theo bất đẳng thức trung bình cộng–trung bình nhân,
$\ln \sum_j e^{z_j} \ge \ln 10 + \frac{1}{10}\sum_j z_j$, với dấu bằng chỉ khi mọi logit bằng nhau.

### Độ dốc của cross-entropy: $\hat y - y$
Để biết nên chỉnh logit theo hướng nào, ta cần đạo hàm của mất mát theo từng logit. Với softmax + cross-entropy,
kết quả gọn đến bất ngờ (chứng minh ở Bài 06, MLSysBook cũng nêu):

$$\frac{\partial \mathcal{L}_\text{CE}}{\partial z_k} = \hat y_k - y_k .$$

Ta kiểm tra bằng **sai phân hữu hạn**: nhích từng logit một chút $\pm\varepsilon$ và xem mất mát đổi bao nhiêu.

In [ ]:
def grad_ce(Z, y):
    """∂𝓛/∂z của cross-entropy trung bình trên các ảnh: (ŷ − y_onehot) / số ảnh."""
    G = softmax(Z)
    G[np.arange(len(y)), y] -= 1
    return G / len(y)


def numeric_grad(f, z, eps=1e-6):
    """Đạo hàm của f theo từng logit bằng sai phân trung tâm: (f(z + ε) − f(z − ε)) / 2ε."""
    g = np.zeros_like(z)
    for k in range(len(z)):
        d = np.zeros_like(z)
        d[k] = eps
        g[k] = (f(z + d) - f(z - d)) / (2 * eps)
    return g


g0 = grad_ce(z0[None], yte[:1])[0]
g0_num = numeric_grad(lambda z: cross_entropy(z[None], yte[:1]), z0)
print("ŷ − y =", "  ".join(f"{k}: {vn(v, 3)}" for k, v in enumerate(g0)))
print("lệch lớn nhất so với sai phân hữu hạn:", sci(np.abs(g0 - g0_num).max()))
assert np.allclose(g0, g0_num, atol=1e-8)

Ở chữ số đúng (số 7), độ dốc là $\hat y_7 - 1 < 0$: tăng logit này làm mất mát giảm. Ở 9 chữ số còn lại, độ dốc là
$\hat y_k > 0$: giảm chúng thì mất mát giảm, và chữ số nào đang được xác suất cao bị đẩy xuống mạnh nhất.
Cross-entropy chỉ *nhìn* $\hat y_c$, nhưng $\hat y_c$ phụ thuộc vào cả 10 logit, nên cả 10 đều nhận một cú đẩy.
Độ dốc mỗi logit luôn nằm trong $(-1, 1)$, kể cả khi mất mát rất lớn.

## 6 · Huấn luyện bằng cross-entropy: dựng lại mạng của demo

Công thức của khóa học: SGD theo mini-batch 64 ảnh, tốc độ học $\eta = 0{,}1$, 5 lượt (epoch) qua 60.000 ảnh,
seed 0, giống hệt chương trình đã tạo ra demo. Mỗi bước tính logit cho 64 ảnh, lấy $\partial\mathcal{L}/\partial z$,
truyền nó ngược qua các lớp (Bài 06), rồi trừ $\eta$ nhân gradient khỏi mọi trọng số (Bài 05). Trong hàm dưới,
chỉ một dòng phụ thuộc vào hàm mất mát: `dz = grad(Z, y)`. Ta sẽ đổi đúng dòng đó ở mục 8.

In [ ]:
def train(seed=0, eta=0.1, epochs=5, grad=grad_ce, last_scale=1.0, X=Xtr, y=ytr, X_eval=Xte, y_eval=yte):
    """SGD theo mini-batch 64 ảnh, như reel/mnist_mlp/train.py. grad(Z, y) trả về ∂𝓛/∂z của hàm mất mát đang dùng."""
    params, rng = init(seed, last_scale)
    log = {"steps": 0, "step_loss": [], "epoch_acc": [], "epoch_loss": []}
    running = []
    for _ in range(epochs):
        order = rng.permutation(len(y))
        for i in range(0, len(order), 64):
            idx = order[i : i + 64]
            h1, h2, Z = forward(params, X[idx])
            running.append(cross_entropy(Z, y[idx]))  # luôn ghi lại CE, kể cả khi mạng học bằng hàm khác
            if len(running) == 50:  # mỗi 50 bước ghi trung bình một lần
                log["step_loss"].append(float(np.mean(running)))
                running = []
            dz = grad(Z, y[idx])  # dòng duy nhất phụ thuộc vào hàm mất mát
            W2, W3 = params[1][0], params[2][0]  # lan truyền ngược (Bài 06) ...
            dh2 = (dz @ W3.T) * (h2 > 0)
            dh1 = (dh2 @ W2.T) * (h1 > 0)
            grads = [(X[idx].T @ dh1, dh1.sum(0)), (h1.T @ dh2, dh2.sum(0)), (h2.T @ dz, dz.sum(0))]
            for layer, (gW, gb) in zip(params, grads):  # ... và một bước gradient descent (Bài 05)
                layer[0] -= eta * gW
                layer[1] -= eta * gb
            log["steps"] += 1
        Z_eval = forward(params, X_eval)[2]
        log["epoch_acc"].append(float((Z_eval.argmax(1) == y_eval).mean()))
        log["epoch_loss"].append(cross_entropy(Z_eval, y_eval))
    return params, log


t0 = time.perf_counter()
params, log = train(seed=0)
print(f"{log['steps']} bước ({vn(time.perf_counter() - t0, 1)} giây)")
for e, (a, l) in enumerate(zip(log["epoch_acc"], log["epoch_loss"]), 1):
    print(f"lượt {e}: kiểm tra đúng {pct(a)}, mất mát {vn(l, 4)}")
acc, loss = log["epoch_acc"][-1], log["epoch_loss"][-1]
print(f"mất mát huấn luyện: 50 bước đầu {vn(log['step_loss'][0], 4)}, 50 bước cuối {vn(log['step_loss'][-1], 4)}")
assert abs(acc - 0.9727) < 5e-4 and abs(loss - 0.0865) < 5e-4, "khác với demo: hãy kiểm tra lại dữ liệu"
assert np.allclose(log["epoch_acc"], [0.9427, 0.9633, 0.9688, 0.9706, 0.9727], atol=1e-3)
assert abs(log["step_loss"][0] - 1.2031) < 1e-3 and abs(log["step_loss"][-1] - 0.059) < 1e-3

Đúng 97,27% trên 10.000 ảnh kiểm tra, mất mát trung bình 0,0865: những con số demo ghi. Hình dưới vẽ mất mát trong
lúc học: đường xanh liền là cross-entropy trên ảnh huấn luyện (trung bình mỗi 50 bước), các ô vuông cam nối nét đứt
là mất mát trên tập kiểm tra sau mỗi lượt, đường xám là mốc $\ln 10$ của việc đoán đều.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
steps = 50 * np.arange(1, len(log["step_loss"]) + 1)
ax.plot(steps, log["step_loss"], color=BLUE, lw=2, label="huấn luyện (mỗi 50 bước)")
per_epoch = log["steps"] // 5 * np.arange(1, 6)
ax.plot(per_epoch, log["epoch_loss"], color=ORANGE, ls="--", marker="s", ms=7, label="kiểm tra (sau mỗi lượt)")
ax.axhline(np.log(10), color=GREY, lw=1, ls=":")
ax.text(steps[-1], np.log(10) - 0.06, "ln 10: đoán đều 10 chữ số", ha="right", va="top", color="#555", fontsize=9)
ax.plot([0], [loss0], marker="o", color=GREY)
ax.annotate(f"chưa học: {vn(loss0, 2)}", (0, loss0), xytext=(120, loss0 - 0.03), fontsize=9, color="#555")
ax.set(xlabel="bước SGD (mỗi bước 64 ảnh)", ylabel="cross-entropy", ylim=(0, 2.6), xlim=(-50, steps[-1] + 50))
ax.set_title("Mất mát khi huấn luyện mạng 784 → 128 → 64 → 10 (η = 0,1, seed 0)")
vn_ticks(ax.yaxis)
vn_ticks(ax.xaxis, 0)
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.3)
plt.show()

Mất mát rơi rất nhanh trong vài trăm bước đầu (trung bình 50 bước đầu tiên đã là 1,2031), rồi giảm chậm dần.
Cuối cùng mất mát huấn luyện thấp hơn mất mát kiểm tra: mạng đã bắt đầu thuộc lòng một chút tập huấn luyện (Bài 08).

## 7 · Softmax trên logit thật của mạng đã học

Giờ ta xem lại 18 ảnh của demo qua mạng vừa huấn luyện. Ô dưới tính logit và xác suất cho cả tập kiểm tra,
kiểm tra rằng mỗi ảnh rơi đúng vào nhóm demo đã xếp (“đúng & chắc”: đúng và $0{,}9 \le \hat y_c < 0{,}999$;
“phân vân”: đúng nhưng $\hat y_c < 0{,}6$; “sai & tự tin”: sai với $\ge 90\%$ cho một chữ số khác), rồi in vài ảnh.

In [ ]:
Z = forward(params, Xte)[2]
P = softmax(Z)
pred, p_true, p_max = Z.argmax(1), P[np.arange(len(yte)), yte], P.max(1)
# Hai ảnh của demo nằm sát ngưỡng (#3: ŷ_c = 0,99897; một ảnh sai có p_max = 0,9006). Trên máy khác, phép cộng
# làm tròn khác đi nên logit có thể lệch ~0,01; TOL cho phép lệch nhỏ đó mà không báo lỗi.
TOL = 0.002
rules = {
    "đúng & chắc": (pred == yte) & (p_true >= 0.9 - TOL) & (p_true < 0.999 + TOL),
    "đúng nhưng phân vân": (pred == yte) & (p_true < 0.6 + TOL),
    "sai & tự tin": (pred != yte) & (p_max >= 0.9 - TOL),
}
for group, idx in DEMO.items():
    assert rules[group][idx].all(), f"nhóm “{group}” khác với demo"
    print(f"{group}:")
    for i in idx[:3]:
        print(
            f"  #{i:>4} số {yte[i]}: mạng chọn {pred[i]} với {pct(p_max[i], 2)}; ŷ của đáp án {pct(p_true[i], 3)}, "
            f"CE = {vn(-np.log(p_true[i]), 4)}"
        )
z_2654 = [-6.47, 10.9505, 1.7756, -0.2557, 0.6202, -2.619, 1.5918, -1.8769, 2.7477, -3.0513]  # logit trong demo
assert np.abs(Z[2654] - z_2654).max() < 0.01, "khác với demo: hãy kiểm tra lại dữ liệu"

Hình dưới đặt ba ảnh cạnh nhau, mỗi hàng một ảnh: logit (giữa) và xác suất sau softmax (phải). Cột xanh là chữ số
đúng; cột cam gạch chéo là chữ số mạng chọn khi nó chọn sai.

In [ ]:
def bars(ax, values, c, wrong, fmt):
    colors = [BLUE if k == c else ORANGE if k == wrong else GREY for k in range(10)]
    hatch = ["//" if k == wrong else "" for k in range(10)]
    ax.barh(range(10), values, color=colors, hatch=hatch, edgecolor="white")
    ax.set_yticks(range(10))
    ax.invert_yaxis()
    for k, v in enumerate(values):
        if k in (c, wrong):
            ax.text(v, k, " " + fmt(v), va="center", ha="left" if v >= 0 else "right", fontsize=8)


fig, axes = plt.subplots(3, 3, figsize=(10, 7.6), gridspec_kw={"width_ratios": [1, 2, 2]})
for row, i in zip(axes, [0, 340, 2654]):
    c, k = yte[i], pred[i]
    row[0].imshow(test_img[i], cmap="gray_r")
    row[0].set_title(f"#{i} · số {c}", fontsize=10)
    row[0].axis("off")
    bars(row[1], Z[i], c, k if k != c else -1, lambda v: vn(v, 1))
    row[1].axvline(0, color="black", lw=0.8)
    row[1].set(xlim=(-14, 17), ylabel="chữ số")
    vn_ticks(row[1].xaxis, 0)
    bars(row[2], P[i], c, k if k != c else -1, lambda v: pct(v, 3 if v < 0.01 else 1))
    row[2].set(xlim=(0, 1.3), xticks=[0, 0.5, 1])
    vn_ticks(row[2].xaxis)
axes[0, 1].set_title("logit z")
axes[0, 2].set_title("xác suất ŷ = softmax(z)")
axes[2, 1].set_xlabel("logit")
axes[2, 2].set_xlabel("xác suất")
plt.tight_layout()
plt.show()

- **#0 (số 7):** logit của 7 vượt xa các chữ số khác, nên gần như cả xác suất dồn về 7. Mất mát gần 0 nhưng chưa
  bằng 0: tăng logit của 7 thêm nữa thì cross-entropy vẫn giảm, dù độ chính xác không đổi.
- **#340 (số 5):** logit của 5 và của 3 gần bằng nhau, nên xác suất chia đôi. Mạng chọn đúng, độ chính xác tính ảnh
  này là đúng hoàn toàn, nhưng cross-entropy vẫn phạt vì mạng phân vân.
- **#2654 (số 6):** mạng chắc gần như tuyệt đối đây là số 1. Xác suất cho số 6 rất nhỏ, nên mất mát rất lớn.

Logit của mạng đã học lớn hơn nhiều so với lúc chưa học. Nhớ ngưỡng tràn số của float16 ở mục 4:

In [ ]:
print(
    f"logit lớn nhất trên tập kiểm tra: {vn(Z.max(), 2)};  ảnh có logit lớn nhất > 11,09: {vn((Z.max(1) > 11.09).sum(), 0)}"
)
Z16 = Z.astype(np.float16)
with np.errstate(over="ignore", invalid="ignore"):
    E16 = np.exp(Z16)
    P_naive = E16 / E16.sum(1, keepdims=True)
print(f"softmax ngây thơ bằng float16: {vn(np.isnan(P_naive).any(1).sum(), 0)} ảnh ra nan")
ce16 = float(-log_softmax(Z16)[np.arange(len(yte)), yte].astype(float).mean())
print(f"log-softmax bằng float16: mất mát {vn(ce16, 4)}  (float64: {vn(loss, 4)})")
assert np.isnan(P_naive).any() and abs(ce16 - loss) < 1e-3

Hơn một nửa tập kiểm tra sẽ làm softmax ngây thơ ra `nan` nếu tính bằng float16. Bản trừ max, cùng kiểu số đó,
cho mất mát gần như bằng float64. Vì vậy thư viện luôn nhận *logit* chứ không nhận xác suất: `cross_entropy`
của PyTorch tính log-softmax bên trong, và khi huấn luyện bằng float16 tự động, PyTorch vẫn tính softmax,
log-softmax và các hàm mất mát bằng float32 ([tài liệu autocast](https://docs.pytorch.org/docs/stable/amp.html)).

## 8 · MSE hay cross-entropy?

Sai số bình phương là hàm mất mát quen thuộc của bài toán hồi quy. Áp vào 10 xác suất (như demo): cộng bình phương
sai lệch trên 10 lớp rồi lấy trung bình trên các ảnh,

$$\mathrm{MSE} = \sum_{k=0}^{9} (\hat y_k - y_k)^2 .$$

Mỗi sai lệch $|\hat y_k - y_k|$ không quá 1 và tổng các $\hat y_k$ bằng 1, nên MSE của một ảnh không bao giờ quá 2
(khi toàn bộ xác suất dồn vào một chữ số sai). Độ dốc theo logit cũng tính được từ đạo hàm của softmax;
theo logit của chữ số đúng, nó gọn thành $\partial \mathrm{MSE}/\partial z_c = -2\,\hat y_c\,\mathrm{MSE}$.

In [ ]:
def mse(Z, y):
    R = softmax(Z)
    R[np.arange(len(y)), y] -= 1  # ŷ − y_onehot
    return float((R**2).sum(1).mean())


def grad_mse(Z, y):
    """∂MSE/∂z_j = 2·ŷ_j·(r_j − Σₖ rₖŷₖ), với r = ŷ − y (đạo hàm của softmax), chia cho số ảnh."""
    P_ = softmax(Z)
    R = P_.copy()
    R[np.arange(len(y)), y] -= 1
    return 2 * P_ * (R - (R * P_).sum(1, keepdims=True)) / len(y)


i = 2654
g_num = numeric_grad(lambda z: mse(z[None], yte[i : i + 1]), Z[i])
assert np.allclose(grad_mse(Z[i : i + 1], yte[i : i + 1])[0], g_num, atol=1e-8)
mse_test = mse(Z, yte)
print(f"MSE trên tập kiểm tra: {vn(mse_test, 4)}   (cross-entropy: {vn(loss, 4)})")
assert abs(mse_test - 0.0411) < 5e-4, "khác với demo: hãy kiểm tra lại dữ liệu"

Bây giờ so hai hàm trên 6 ảnh “sai & tự tin” của demo: mất mát, và độ dốc theo logit của chữ số đúng, tức là cú đẩy
mà mỗi hàm cho logit đó.

In [ ]:
print(" ảnh   đúng→chọn      ŷ_c       CE     MSE   dốc CE    dốc MSE   CE/MSE")
for i in DEMO["sai & tự tin"]:
    z, c = Z[i : i + 1], yte[i : i + 1]
    d_ce, d_mse = grad_ce(z, c)[0, c[0]], grad_mse(z, c)[0, c[0]]
    m = mse(z, c)
    assert np.isclose(d_mse, -2 * p_true[i] * m)  # công thức gọn ở trên
    print(
        f"#{i:>4}   {yte[i]} → {pred[i]}   {sci(p_true[i]):>9}  {vn(cross_entropy(z, c), 2):>6}  {vn(m, 4)}  "
        f"{vn(d_ce, 4):>7}  {vn(d_mse, 5):>9}  {vn(d_ce / d_mse, 0):>6} lần"
    )

Cả 6 ảnh có MSE sát mức trần 2, nên MSE gần như không phân biệt được “sai nặng” với “sai rất nặng”.
Cross-entropy thì từ 5,86 đến 9,36 và không có trần. Quan trọng hơn là độ dốc: với cross-entropy, logit của chữ số
đúng nhận cú đẩy gần $-1$, mức lớn nhất có thể. Với MSE, cú đẩy nhỏ hơn từ 88 đến 2.903 lần, vì
$-2\,\hat y_c\,\mathrm{MSE}$ có thừa số $\hat y_c \approx 0$.

Hình dưới kéo logit của chữ số đúng (số 6) của ảnh #2654 từ rất thấp đến rất cao, giữ nguyên 9 logit còn lại,
như demo làm. Bên trái là mất mát: cross-entropy (xanh, liền), MSE (cam, đứt) và sai số 0–1 (xanh lá, chấm: 1 khi
đoán sai, 0 khi đúng). Bên phải là độ lớn độ dốc theo logit đó, trục dọc theo thang log.

In [ ]:
c = yte[2654]
grid = np.linspace(-10, 20, 601)
Zg = np.repeat(Z[2654][None], len(grid), axis=0)
Zg[:, c] = grid
yg = np.full(len(grid), c)
ce_g = -log_softmax(Zg)[:, c]
R = softmax(Zg) - np.eye(10)[c]
mse_g = (R**2).sum(1)
rival = np.delete(Z[2654], c).max()  # logit lớn nhất của 9 chữ số còn lại
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.6))
a.plot(grid, ce_g, color=BLUE, lw=2.2, label="cross-entropy")
a.plot(grid, mse_g, color=ORANGE, lw=2, ls="--", label="MSE")
a.plot(grid, (grid < rival).astype(float), color=GREEN, lw=2, ls=":", label="sai số 0–1")
b.semilogy(grid, np.abs(grad_ce(Zg, yg)[:, c] * len(grid)), color=BLUE, lw=2.2, label="cross-entropy")
b.semilogy(grid, np.abs(grad_mse(Zg, yg)[:, c] * len(grid)), color=ORANGE, lw=2, ls="--", label="MSE")
for ax in (a, b):
    ax.axvline(Z[2654, c], color="black", lw=1)
    ax.text(Z[2654, c] + 0.4, 0.97, "mạng thật", transform=ax.get_xaxis_transform(), va="top", fontsize=9)
    ax.set_xlabel(f"logit của chữ số đúng ({c}); 9 logit còn lại giữ nguyên")
    vn_ticks(ax.xaxis, 0)
    ax.legend(frameon=False, fontsize=9)
a.set(ylabel="mất mát", ylim=(0, 12), title=f"Ảnh #2654: mất mát theo logit của số {c}")
vn_ticks(a.yaxis, 0)
b.set(ylabel="|độ dốc| (thang log)", ylim=(1e-9, 3), title="Độ lớn cú đẩy cho logit đó")
plt.tight_layout()
plt.show()

Bên trái: khi logit của số 6 còn thấp hơn logit của số 1, cross-entropy tăng gần như theo đường thẳng (mỗi đơn vị
logit thêm khoảng 1 nat), còn MSE nằm phẳng sát mức 2. Bên phải: độ dốc của cross-entropy giữ gần 1 suốt vùng
đoán sai, còn độ dốc của MSE rơi theo hàm mũ khi logit giảm. Sai số 0–1 thì phẳng hoàn toàn ở mọi chỗ trừ một
bậc nhảy: nó không cho gradient nào, nên không dùng để học được, dù chính nó là thứ ta muốn giảm.

### Một mạng bắt đầu tự tin mà sai
Trên mạng đã học, rất ít ảnh rơi sâu vào vùng phẳng của MSE (ô dưới đếm). Muốn thấy hậu quả khi rất nhiều ảnh
nằm ở đó, ta khởi tạo lớp ra lớn gấp 30 lần: logit ngẫu nhiên tản mạn đến mức mạng chưa học đã “chắc chắn” về một
chữ số ngẫu nhiên cho phần lớn các ảnh. Ô dưới đếm số ảnh có $\hat y_c < 0{,}001$ lúc bắt đầu.

In [ ]:
print(f"mạng đã học ở mục 6: {(p_true < 1e-3).sum()} trên {vn(len(yte), 0)} ảnh kiểm tra có ŷ_c < 0,001")
params30, _ = init(seed=0, last_scale=30)
Z30 = forward(params30, Xtr)[2]
stuck0 = (softmax(Z30)[np.arange(len(ytr)), ytr] < 1e-3).sum()
print(f"W₃ × 30: mất mát ban đầu {vn(cross_entropy(forward(params30, Xte)[2], yte), 2)} (so với ln 10 = 2,30)")
print(f"{vn(stuck0, 0)} trên {vn(len(ytr), 0)} ảnh huấn luyện có ŷ_c < 0,001 lúc bắt đầu ({pct(stuck0 / len(ytr), 1)})")

### 🤔 Dự đoán trước
Huấn luyện mạng này một lượt (cùng η = 0,1, cùng 60.000 ảnh), một lần bằng cross-entropy, một lần bằng MSE.
Sau đó, bao nhiêu trong 10.000 ảnh kiểm tra vẫn có $\hat y_c < 0{,}001$? Hàng chục, hàng trăm hay hàng nghìn?

In [ ]:
stuck = {"ce": [], "mse": []}
for name, g in [("ce", grad_ce), ("mse", grad_mse)]:
    for seed in range(3):
        p_s, log_s = train(seed=seed, epochs=1, grad=g, last_scale=30)
        pc = softmax(forward(p_s, Xte)[2])[np.arange(len(yte)), yte]
        stuck[name].append(int((pc < 1e-3).sum()))
        print(
            f"{name:>3}, seed {seed}: kiểm tra đúng {pct(log_s['epoch_acc'][0], 2)}, "
            f"{vn(stuck[name][-1], 0):>5} ảnh vẫn có ŷ_c < 0,001"
        )
print(f"MSE để lại ít nhất {vn(min(stuck['mse']) / max(stuck['ce']), 0)} lần nhiều ảnh kẹt hơn cross-entropy")
assert max(stuck["ce"]) * 10 < min(stuck["mse"])

Sau một lượt, cross-entropy chỉ để lại 57–73 ảnh kiểm tra còn kẹt ở $\hat y_c < 0{,}001$. MSE để lại 3.512–4.122 ảnh,
nhiều gấp ít nhất 48 lần: khoảng cách lớn hơn nhiều so với dao động giữa các seed. Độ chính xác của MSE cũng thấp hơn ở cả ba
seed (57,22–63,49% so với 65,71–89,41%), dù cross-entropy dao động rất mạnh giữa các seed. Khởi tạo quá tự tin cũng
làm hại cross-entropy (lượt đầu ở mục 6 đạt 94,27%), nhưng nó thoát ra được.

Bridle (1990) đã chỉ ra điều này, và sách *Deep Learning* (Goodfellow và cộng sự, mục 6.2.2.3) nhắc lại: với
softmax, sai số bình phương có thể không sửa được mạng ngay cả khi nó sai một cách rất tự tin, vì không có logarit
để “hoàn tác” hàm mũ. Cross-entropy có: $-\ln \hat y_c = \ln \sum_j e^{z_j} - z_c$ gần như tuyến tính theo logit
khi mạng sai, nên độ dốc không bao giờ tắt.

## 9 · Xác suất của softmax đáng tin đến đâu?

Softmax cho ra những con số trông như xác suất. Nếu mạng nói 90%, nó có đúng khoảng 90% số lần không? Ô dưới chia
10.000 ảnh kiểm tra theo xác suất lớn nhất (độ “tự tin” của mạng), rồi so độ tự tin trung bình với tỉ lệ đúng thật
trong từng nhóm.

In [ ]:
correct = pred == yte
edges = [0, 0.5, 0.6, 0.7, 0.8, 0.9, 0.99, 1.0]
bins = []
for lo, hi in itertools.pairwise(edges):
    m = (p_max >= lo) & ((p_max < hi) | (hi == 1.0))
    bins.append((lo, hi, int(m.sum()), p_max[m].mean(), correct[m].mean()))
    print(
        f"tự tin {pct(lo, 0):>4}–{pct(hi, 0):<4}: {vn(m.sum(), 0):>5} ảnh, tự tin trung bình {pct(p_max[m].mean())}, đúng {pct(correct[m].mean())}"
    )
sure, wrong_sure = p_max >= 0.99, ~correct & (p_max >= 0.9)
print(f"cả tập: tự tin trung bình {pct(p_max.mean())}, đúng {pct(correct.mean())}")
print(
    f"{vn(sure.sum(), 0)} ảnh được ≥ 99%, trong đó đúng {pct(correct[sure].mean())}; "
    f"{wrong_sure.sum()} trong {(~correct).sum()} lỗi có ≥ 90% cho một chữ số sai"
)
# số đếm trên một mạng huấn luyện bằng số thực: cho phép lệch vài ảnh giữa các máy (BLAS làm tròn khác nhau)
assert abs(sure.sum() - 7799) <= 5 and abs(correct[sure].mean() - 0.9986) < 1e-3, "khác với demo"
assert abs(wrong_sure.sum() - 52) <= 5 and abs((~correct).sum() - 273) <= 5, "khác với demo"

Hình dưới vẽ bảng trên: mỗi chấm tròn xanh là một nhóm, trục ngang là độ tự tin trung bình, trục dọc là tỉ lệ đúng,
số cạnh chấm là số ảnh. Đường chéo xám nét đứt là “nói 90% thì đúng 90%”.

In [ ]:
fig, ax = plt.subplots(figsize=(4.8, 4.6))
ax.plot([0, 1], [0, 1], color=GREY, ls="--", lw=1)
cx, cy = [b[3] for b in bins], [b[4] for b in bins]
ax.scatter(cx, cy, s=[30 + 120 * np.sqrt(b[2] / len(yte)) for b in bins], color=BLUE, zorder=3)
for b in bins:
    # hai chấm cuối nằm sát nhau ở góc trên bên phải: nhãn của chấm cuối đặt bên dưới, lệch phải
    last = b[3] > 0.99
    ax.annotate(
        f"{vn(b[2], 0)} ảnh",
        (b[3], b[4]),
        xytext=(16, -18) if last else (-8, 6),
        textcoords="offset points",
        ha="center" if last else "right",
        fontsize=8,
    )
ax.set(xlim=(0.3, 1.02), ylim=(0.3, 1.02), xlabel="độ tự tin trung bình (xác suất lớn nhất)", ylabel="tỉ lệ đúng")
ax.set_title("Độ tự tin và tỉ lệ đúng\n(10.000 ảnh kiểm tra)", fontsize=10)
vn_ticks(ax.xaxis)
vn_ticks(ax.yaxis)
ax.set_aspect("equal")
plt.show()

Trên ảnh chữ số thật, các chấm nằm sát đường chéo: độ tự tin trung bình 97,15%, tỉ lệ đúng 97,27%. Mạng nhỏ này
“trung thực” trên loại dữ liệu nó đã học. (Guo và cộng sự, 2017, thấy các mạng hiện đại lớn hơn nhiều thường quá
tự tin: chấm nằm dưới đường chéo.) Nhưng 52 lỗi vẫn được mạng cho ≥ 90%: độ tự tin cao không bảo đảm đúng.

### 🤔 Dự đoán trước
Đưa cho mạng 1.000 ảnh **nhiễu**: mỗi điểm ảnh là một số ngẫu nhiên đều trong $[0, 1]$, không hề là chữ số.
Mạng chưa bao giờ thấy ảnh như vậy. Xác suất lớn nhất của nó sẽ quanh 1/10 (“không biết”), hay vẫn cao?

In [ ]:
noise = np.random.default_rng(0).random((1000, 784))
Pn = softmax(forward(params, noise)[2])
print(
    f"{vn(len(noise), 0)} ảnh nhiễu: xác suất lớn nhất trung bình {pct(Pn.max(1).mean())};  {pct((Pn.max(1) >= 0.9).mean(), 1)} số ảnh nhiễu được ≥ 90%"
)
print("chữ số được chọn:", {k: int(n) for k, n in enumerate(np.bincount(Pn.argmax(1), minlength=10)) if n})
fig, axes = plt.subplots(1, 5, figsize=(8, 1.9))
for ax, j in zip(axes, np.argsort(-Pn.max(1))[:5]):
    ax.imshow(noise[j].reshape(28, 28), cmap="gray_r", vmin=0, vmax=1)
    ax.set_title(f"“số {Pn[j].argmax()}”: {pct(Pn[j].max(), 1)}", fontsize=9)
    ax.axis("off")
plt.suptitle("5 ảnh nhiễu mạng tự tin nhất", fontsize=10, y=1.04)
plt.show()

Mạng không trả lời “không biết”. Softmax luôn chia đủ 100% cho 10 chữ số, vì không có lựa chọn “không phải chữ số”,
và trên ảnh nhiễu mạng vẫn thường khá tự tin (hình trên: những ảnh nhiễu nó tự tin nhất). Trên ảnh nhiễu, độ tự tin
trung bình thấp hơn trên ảnh thật (70,45% so với 97,15%), nên Hendrycks & Gimpel (2017) dùng chính xác suất lớn nhất làm mốc cơ bản
để phát hiện ảnh lạ, nhưng không thể dựa vào nó: xác suất của softmax chỉ có nghĩa trên loại dữ liệu mạng đã học.

## 10 · Mất mát là một “địa hình”

Mất mát là một hàm của cả 109.386 tham số. Không vẽ được không gian ấy, nhưng ta có thể đi trên một đường thẳng
trong đó: từ trọng số lúc khởi tạo $\theta_0$ đến trọng số đã học $\theta_1$,
$\theta(\alpha) = (1-\alpha)\,\theta_0 + \alpha\,\theta_1$, và đo mất mát dọc đường (cách nhìn của
Goodfellow, Vinyals & Saxe, 2015). Hình dưới: trên là cross-entropy (xanh, chấm tròn), dưới là độ chính xác
(cam, ô vuông, nét đứt), cả hai trên 10.000 ảnh kiểm tra.

In [ ]:
alphas = np.linspace(-0.25, 1.5, 15)
path_loss, path_acc = [], []
for a in alphas:
    theta = [[(1 - a) * W0 + a * W1, (1 - a) * b0 + a * b1] for (W0, b0), (W1, b1) in zip(params0, params)]
    Za = forward(theta, Xte)[2]
    path_loss.append(cross_entropy(Za, yte))
    path_acc.append(float((Za.argmax(1) == yte).mean()))
i0, i1 = np.argmin(np.abs(alphas)), np.argmin(np.abs(alphas - 1))
print("α:      ", "  ".join(f"{vn(a, 3):>6}" for a in alphas))
print("mất mát:", "  ".join(f"{vn(v, 3):>6}" for v in path_loss))
print(
    f"từ θ₀ đến θ₁, mất mát giảm ở mọi bước: {bool(np.all(np.diff(path_loss[i0 : i1 + 1]) < 0))};  "
    f"nhỏ nhất ở α = {vn(alphas[np.argmin(path_loss)], 3)}"
)
assert np.isclose(path_loss[i0], loss0) and np.isclose(path_loss[i1], loss)
fig, (a, b) = plt.subplots(2, 1, figsize=(8, 4.6), sharex=True)
a.plot(alphas, path_loss, color=BLUE, marker="o", lw=2)
b.plot(alphas, 100 * np.array(path_acc), color=ORANGE, marker="s", ls="--", lw=2)
for ax in (a, b):
    for x0 in (0, 1):
        ax.axvline(x0, color=GREY, lw=1)
a.text(0.02, 2.2, "θ₀: lúc khởi tạo", fontsize=9)
a.text(1.02, 2.2, "θ₁: đã học", fontsize=9)
a.set(ylabel="cross-entropy", title="Đi thẳng từ trọng số ban đầu đến trọng số đã học")
b.set(ylabel="đúng (%)", xlabel="α  (θ = (1 − α)·θ₀ + α·θ₁)")
vn_ticks(a.yaxis)
vn_ticks(b.xaxis, 2)
for ax in (a, b):
    ax.grid(alpha=0.3)
plt.show()

Trên đường thẳng này, mất mát giảm đều từ $\theta_0$ đến $\theta_1$, và độ chính xác tăng vọt ở đoạn giữa. Đi quá
$\theta_1$ một chút ($\alpha > 1$), mất mát lại tăng: $\theta_1$ nằm gần đáy của lát cắt này, dù chưa chắc là đáy
thật của cả không gian. Gradient descent (Bài 05) không biết trước $\theta_1$. Nó chỉ đứng ở chỗ hiện tại, nhìn độ
dốc $\partial\mathcal{L}/\partial\theta$ và bước xuống. Hàm mất mát chính là địa hình nó đi trên.

## 11 · Thử thay đổi

Các ô dưới chạy lại cùng các hàm với tham số khác. Bạn có thể sửa giá trị rồi chạy lại.

### Nhiệt độ của softmax
Chia mọi logit cho một số $T > 0$ trước softmax: $T < 1$ làm xác suất “nhọn” hơn (tự tin hơn), $T > 1$ làm chúng
“phẳng” hơn. Thứ tự các chữ số và độ chính xác không đổi, chỉ có độ tự tin đổi. Đây cũng là nút “nhiệt độ” khi
GPT-2 sinh chữ (Bài 17). Ô dưới đo cross-entropy kiểm tra của mạng đã học ở nhiều nhiệt độ, và xác suất cho
chữ số mạng chọn ở ảnh phân vân #340.

In [ ]:
for T in [0.5, 0.75, 1, 1.5, 2, 4]:
    ZT = Z / T
    assert ((ZT.argmax(1) == yte).mean()) == acc  # độ chính xác không đổi
    print(
        f"T = {vn(T, 2):>4}: mất mát kiểm tra {vn(cross_entropy(ZT, yte), 4)};  ảnh #340: ŷ của số 5 = {pct(softmax(ZT[340])[5], 1)}"
    )

Mất mát nhỏ nhất ở đúng $T = 1$ trong các giá trị này: mạng đã “hiệu chỉnh” sẵn, như hình ở mục 9 cho thấy.
Với các mạng lớn quá tự tin, Guo và cộng sự (2017) chọn $T > 1$ trên tập kiểm định (*temperature scaling*), và
chỉ một tham số đó là đủ sửa phần lớn độ quá tự tin.

### MSE có thật sự không học được không?
Mục 8 cho thấy MSE thua khi mạng bắt đầu tự tin mà sai. Còn với khởi tạo bình thường thì sao? Độ dốc của MSE nhỏ
hơn, nên công bằng là cho nó tốc độ học lớn hơn. Ô dưới huấn luyện một lượt, 3 seed, và đo trên **tập kiểm định**
(5.000 ảnh cuối của tập huấn luyện, `train[55k:]`, như các bài sau của khóa học), không phải tập kiểm tra,
vì ta đang chọn $\eta$.

In [ ]:
Xa, ya, Xv, yv = Xtr[:55000], ytr[:55000], Xtr[55000:], ytr[55000:]
print(f"huấn luyện trên {vn(len(ya), 0)} ảnh, kiểm định trên {vn(len(yv), 0)} ảnh")
val = {}
for name, g, eta in [("cross-entropy", grad_ce, 0.1), ("MSE", grad_mse, 0.1), ("MSE", grad_mse, 1.0)]:
    r = [
        train(seed=s, eta=eta, epochs=1, grad=g, X=Xa, y=ya, X_eval=Xv, y_eval=yv)[1]["epoch_acc"][0] for s in range(3)
    ]
    val[(name, eta)] = r
    print(
        f"{name:>13}, η = {vn(eta, 1)}: kiểm định đúng {' · '.join(pct(a) for a in r)}  (từ {pct(min(r))} đến {pct(max(r))})"
    )

Ở cùng $\eta = 0{,}1$, MSE thấp hơn cross-entropy ở cả ba seed (cùng seed thì cùng khởi tạo và cùng thứ tự ảnh),
nhưng hai khoảng chồng lên nhau. Với $\eta = 1$, khoảng của MSE (91,00–96,30%) bao trùm khoảng của cross-entropy:
chênh lệch nằm trong dao động giữa các seed. (Ở đây ta chưa dò $\eta$ cho cross-entropy; thử thêm vào danh sách.) Vậy “MSE không học được phân loại” là nói quá. Hui & Belkin
(2021) huấn luyện nhiều mạng lớn bằng MSE và thấy nó thường ngang cross-entropy. Lý do dùng cross-entropy mặc định là
nó không bao giờ “tắt” độ dốc khi mạng sai mà tự tin, và nó là log-likelihood, nên $\hat y$ có nghĩa xác suất.

Thử thêm: đổi `last_scale` ở mục 8 thành 10 hoặc 100; đổi `eta`; hay thay `grad_mse` bằng một hàm mất mát của
riêng bạn (chỉ cần trả về $\partial\mathcal{L}/\partial z$).

## 12 · Giới hạn và cầu nối sang bài sau

- **Mất mát mới chỉ nói sai bao nhiêu.** Để sửa, mạng cần biết mỗi trọng số phải nhích theo hướng nào: gradient,
  tính bằng lan truyền ngược (Bài 06), rồi đi từng bước xuống dốc (Bài 05). Hàm `train()` ở trên đã dùng cả hai
  như hộp đen.
- **Cross-entropy không bao giờ bằng 0.** Ảnh #0 đã đúng chắc chắn mà vẫn có mất mát dương, nên huấn luyện tiếp
  sẽ đẩy logit lớn mãi. Khi mạng đúng hết tập huấn luyện, điều đó dẫn đến quá tự tin và quá khớp (Bài 08).
- **Cùng một hàm cho mọi bài phân loại.** GPT-2 (Bài 17) được huấn luyện bằng đúng cross-entropy này, mỗi vị trí
  là một bài phân loại token tiếp theo trên 50.257 lớp
  ([Radford và cộng sự, 2019](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)). Mốc “đoán đều” của nó là:

In [ ]:
V = 50257  # số token trong từ điển của GPT-2
print(f"ln {vn(V, 0)} = {vn(np.log(V), 2)} nat cho mỗi token  (MNIST: ln 10 = {vn(np.log(10), 2)})")
Zv = np.random.default_rng(0).normal(size=(4, V))  # logit ngẫu nhiên của 4 "vị trí", chỉ để thử log-softmax
assert np.allclose(np.exp(log_softmax(Zv)).sum(1), 1)

Log-softmax của ta chạy nguyên vẹn trên 50.257 lớp: không gì trong nó phụ thuộc vào số 10.

## Tóm tắt
- Logit là 10 điểm số thô; chia cho tổng không cho ra xác suất (có số âm, có thể đảo thứ tự). Softmax lũy thừa rồi
  chia cho tổng: 10 số dương, tổng 1, giữ thứ tự.
- Softmax chỉ phụ thuộc vào hiệu giữa các logit, nên trừ max trước khi lũy thừa là miễn phí và không bao giờ tràn
  số; với float16, softmax ngây thơ ra `nan` trên hơn nửa tập kiểm tra.
- Cross-entropy $= -\ln \hat y_c$, tính thẳng từ logit bằng log-softmax. Mạng chưa học: 2,4257, gần $\ln 10$;
  độ dốc theo logit là $\hat y - y$.
- Huấn luyện bằng cross-entropy (SGD, 5 lượt): 97,27% trên tập kiểm tra, mất mát 0,0865, đúng như demo.
- Trên ảnh tự tin mà sai, MSE kẹt gần mức trần 2 và độ dốc nhỏ hơn 88 đến 2.903 lần; khởi tạo quá tự tin
  thì MSE để lại hàng nghìn ảnh kẹt, cross-entropy chỉ vài chục. Với khởi tạo tốt và $\eta$ lớn hơn, MSE học được.
- Xác suất của mạng này khớp tỉ lệ đúng trên ảnh thật, nhưng 52 lỗi vẫn ≥ 90%, và ảnh nhiễu cũng nhận độ tự tin cao.

## Câu hỏi ôn tập
1. Vì sao không dùng $z_k / \sum_j z_j$ làm xác suất?
2. Cộng 100 vào cả 10 logit thì softmax đổi thế nào? Còn nhân cả 10 logit với 3?
3. Một mạng phân loại 10 lớp vừa khởi tạo có mất mát ban đầu 9. Điều đó gợi ý gì?
4. Mạng gán 0,0001 cho chữ số đúng. Cross-entropy của ảnh đó là bao nhiêu, và MSE tối đa là bao nhiêu?
5. Vì sao độ dốc của MSE theo logit của chữ số đúng gần 0 khi mạng sai mà tự tin, còn của cross-entropy thì không?
6. Mạng nói 99% cho một ảnh nhiễu. Vì sao softmax không thể nói “không biết”?

<details><summary>Gợi ý đáp án</summary>

1. Logit có thể âm: thương có thể âm hoặc lớn hơn 1, và nếu tổng âm thì thứ tự bị đảo (mục 3).
2. Cộng: không đổi gì, vì $e^{100}$ triệt tiêu giữa tử và mẫu. Nhân 3: hiệu giữa các logit giãn ra, xác suất lớn nhất
   tiến về 1, nhưng thứ tự giữ nguyên.
3. Đoán đều cho $\ln 10 \approx 2{,}30$. Mất mát 9 nghĩa là logit ban đầu quá lớn (lớp ra khởi tạo quá “tự tin”),
   hoặc có lỗi trong mã; nên kiểm tra trước khi huấn luyện.
4. $-\ln 0{,}0001 = 4 \ln 10 \approx 9{,}21$. MSE không quá 2 với mọi ảnh.
5. $\partial\mathrm{MSE}/\partial z_c = -2\,\hat y_c\,\mathrm{MSE}$ có thừa số $\hat y_c \approx 0$; còn
   $\partial\mathcal{L}_\text{CE}/\partial z_c = \hat y_c - 1 \approx -1$, vì logarit hoàn tác hàm mũ.
6. Softmax luôn chia đủ 100% cho 10 lớp đã định; không có lớp “không phải chữ số”, và mạng chỉ học trên chữ số.
</details>

## Tìm hiểu thêm
- MLSysBook, [*Neural Computation*](https://mlsysbook.ai/vol1/nn_computation/nn_computation.html), mục *Loss functions*:
  cross-entropy cho MNIST, mẹo log-sum-exp, mất mát ban đầu gần $\log 10$.
- Goodfellow, Bengio & Courville, [*Deep Learning*, mục 6.2](https://www.deeplearningbook.org/contents/mlp.html):
  đầu ra softmax, vì sao log-likelihood hợp với nó, tên “softargmax”.
- [Dive into Deep Learning, *Softmax Regression*](https://d2l.ai/chapter_linear-classification/softmax-regression.html):
  cross-entropy theo lý thuyết thông tin (nat, mức bất ngờ).
- M. Nielsen, [*Neural Networks and Deep Learning*, Chương 3](http://neuralnetworksanddeeplearning.com/chap3.html):
  sai số bình phương làm chậm việc học khi mạng sai nặng.
- J. S. Bridle (1990), [Probabilistic interpretation of feedforward classification network outputs](https://doi.org/10.1007/978-3-642-76153-9_28).
- CS231n, [Linear classification](https://cs231n.github.io/linear-classify/): ổn định số của softmax;
  A. Karpathy, [A Recipe for Training Neural Networks](https://karpathy.github.io/2019/04/25/recipe/): kiểm tra mất mát lúc khởi tạo.
- C. Guo và cộng sự (2017), [On Calibration of Modern Neural Networks](https://arxiv.org/abs/1706.04599).
- D. Hendrycks & K. Gimpel (2017), [A Baseline for Detecting Misclassified and Out-of-Distribution Examples](https://arxiv.org/abs/1610.02136).
- L. Hui & M. Belkin (2021), [Square Loss vs Cross-Entropy in Classification Tasks](https://arxiv.org/abs/2006.07322).
- I. Goodfellow, O. Vinyals & A. Saxe (2015), [Qualitatively characterizing neural network optimization problems](https://arxiv.org/abs/1412.6544).

**Bài tiếp theo:** [Bài 05 · Gradient descent](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/gradient-descent/):
đi xuống địa hình mất mát từng bước, theo độ dốc.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")